# Getting the Dataset

In [ ]:
def get_IAM_Dataset () :
  import kagglehub
  # Download IAM Handwritten Forms dataset
  path = kagglehub.dataset_download("naderabdalghani/iam-handwritten-forms-dataset")
  print("Dataset downloaded to:", path)
  return path

# Tools and Dependencies

In [ ]:
def import_dependencies () :
  global os, shutil, itertools, np, tf, Input, Conv2D, MaxPooling2D, Dense, Dropout, Flatten, Lambda, Model, plt, accuracy_score, train_test_split, cv2
  import os
  import shutil
  import itertools
  import numpy as np
  import tensorflow as tf
  from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dense, Dropout, Flatten, Lambda
  from tensorflow.keras.models import Model
  import matplotlib.pyplot as plt
  from sklearn.metrics import accuracy_score
  from sklearn.model_selection import train_test_split
  import cv2
import_dependencies()

# Global Parameters

In [ ]:
def init_global_params () :
  global ORIGINAL_DATASET_PATH, PREPROCESSED_PATH, PAIR_DATASET_PATH
  global IMAGE_X, IMAGE_Y, POSITIVE_PAIRS, NEGATIVE_PAIRS, BATCH_SIZE, EPOCHS
  global GLOBAL_RANDOM_STATE, TEST_SET_RATIO, VAL_SET_RATIO, LETTER_PIXEL_THRESHOLD
  path = get_IAM_Dataset()
  ORIGINAL_DATASET_PATH = path + "/data"
  PREPROCESSED_PATH = "/content/Preprocessed"
  PAIR_DATASET_PATH = "/content/Pairs"
  IMAGE_X = 512
  IMAGE_Y = 512
  POSITIVE_PAIRS = 50000  # Maximum Possible without dataset augmentation = 4051 Pairs
  NEGATIVE_PAIRS = 50000
  BATCH_SIZE = 16
  EPOCHS = 10
  GLOBAL_RANDOM_STATE = 42 # To maintain working environment result reproducibility
  TEST_SET_RATIO = 0.20 # Percentage Size of testing set out of full dataset.
  VAL_SET_RATIO = 0.10 # Percentage Size of validation set out of training set.
  LETTER_PIXEL_THRESHOLD = 240
init_global_params()

100%|██████████| 4.31G/4.31G [00:21<00:00, 211MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/naderabdalghani/iam-handwritten-forms-dataset/versions/1


# Analyzing and Preprocessing Dataset

Defining Dataset Preprocessing Utility functions

In [ ]:
# Defining Required Preprocessing per Image

# Remove unwanted printed parts and blank parts along the borders of forms images.
def cropUnwanted (img) :
  # img_size : 3542 * 2479 ,  wanted, 600-2800, 100-2400
  cropped = tf.image.crop_to_bounding_box(img, offset_height=800, offset_width=100, target_height=2200, target_width=2300)
  return cropped

def preprocess (img) :
  img = cropUnwanted(img) # cut out unwanted parts of image
  img = tf.image.rgb_to_grayscale(img) # convert to grayscale (color not needed)
  img = tf.image.resize(img, [IMAGE_X, IMAGE_Y]) # Resize (to save memory)
  img = tf.cast(img, tf.float32) / 255.0 # normalize (to keep values with 0.0-1.0)
  img = tf.clip_by_value(img, 0.0, 1.0) # clip values (to ensure they are in 0.0-1.0)
  return img

In [ ]:
# Dataset augmentation by adding photos that simulate real-life conditions (Left rotation, Right rotation, average blur, gaussian blur)

# Function that rotates images
def rotate_image(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    return rotated

# Function that does average blur
def A_blur_image (img, ksize) :
    return cv2.blur(img, ksize, 0)

# Function that does Gaussian blur
def G_blur_image (img, ksize) :
    return cv2.GaussianBlur(img, ksize, 0)

def generate_augmented_images (img) :
    aug_images = []
    aug_images.append(rotate_image(img,+2))
    aug_images.append(rotate_image(img,-2))
    aug_images.append(A_blur_image(img,(3,3)))
    return aug_images


In [ ]:
# Step 1: Preprocess all images and save in Numpy Array Format for Training
def preprocess_dataset () :
  os.makedirs(PREPROCESSED_PATH, exist_ok=True)

  for folder in os.listdir(ORIGINAL_DATASET_PATH) : # Go through each folder in dataset
      folder_path = os.path.join(ORIGINAL_DATASET_PATH, folder) # Get the path to folder
      save_dir = os.path.join(PREPROCESSED_PATH, os.path.basename(folder_path)) # Specifying the path to new directory to save images
      os.makedirs(save_dir, exist_ok=True) # Creating the new directory to save preprocessed images
      for file in os.listdir(folder_path) : # Go through each image in folder
        if file.endswith(".png") : # Process only ".png" image files
          img_path = os.path.join(folder_path, file) # Get image file path
          org_img = cv2.imread(img_path) # Read the image file into a variable (png -> numpy array) in BGR color order.
          org_img = cv2.cvtColor(org_img, cv2.COLOR_BGR2RGB) # TensorFlow assumes RGB ordering, so converting from BGR to RGB helps if you visualize later.
          all_imgs = [org_img] # List to store all augmented images generated next.
          all_imgs.extend(generate_augmented_images(org_img)) # Store all augmented imagesof original image. (blurred, rotated etc.)
          for i,img in enumerate(all_imgs) : # Preprocess all images including augmented images.
            img = tf.convert_to_tensor(img, dtype=tf.uint8) # Convert numpy array form of augmented images into a tensor for preprocessing
            preprocessed_img = preprocess(img) # Preprocess the image according to defined preprocessing
            save_path = os.path.join(save_dir, f"{file}-{i}.npy") # Specify path to save preprocessed image
            np.save(save_path, preprocessed_img.numpy()) # Save the preprocessed image as a Tensor (.npy)


In [ ]:
# Organising Dataset into Image Pairs (Using thier paths only to save space, not file itself) with label for same writer or not to feed into Model
def generate_data_pairs_by_path (p, n, TEST_SIZE) : # p = number of postive pairs, n = number of negative pairs

    os.makedirs(PAIR_DATASET_PATH, exist_ok=True)
    folders = sorted([f for f in os.listdir(PREPROCESSED_PATH) if os.path.isdir(os.path.join(PREPROCESSED_PATH, f))])

    positive_pairs = []
    negative_pairs = []

    # Generate positive pairs in sorted order (recreate same 'n' pairs always)
    for folder in folders:
        folder_path = os.path.join(PREPROCESSED_PATH, folder)
        files = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".npy")]) # Create paths for all files

        # All combinations with replacement (including self-pairs)
        folder_pairs = list(itertools.combinations_with_replacement(files, 2))

        # Add to positive pairs until we reach n
        for pair in folder_pairs:
            if len(positive_pairs) < p:
                positive_pairs.append(pair)
            else:
                break
        if len(positive_pairs) >= p:
            break

    positive_labels = [1]*len(positive_pairs)

    # Generate negative pairs in sorted order (recreate same 'n' pairs always)
    all_files = [sorted([os.path.join(PREPROCESSED_PATH, f, img)
                        for img in os.listdir(os.path.join(PREPROCESSED_PATH, f)) if img.endswith(".npy")])
                for f in folders]

    for i in range(len(all_files)):
        for j in range(i+1, len(all_files)):
            for f1 in all_files[i]:
                for f2 in all_files[j]:
                    if len(negative_pairs) < n:
                        negative_pairs.append((f1, f2))
                    else:
                        break
                if len(negative_pairs) >= n:
                    break
            if len(negative_pairs) >= n:
                break
        if len(negative_pairs) >= n:
            break

    negative_labels = [0]*len(negative_pairs)

    # Combine in Order
    pairs = positive_pairs + negative_pairs
    labels = positive_labels + negative_labels

    # X = image pair paths , y = True/False label
    X_train, X_test, y_train, y_test = train_test_split(pairs, labels, test_size=TEST_SIZE, random_state=GLOBAL_RANDOM_STATE)

    # Save path pairs and labels
    return X_train, X_test, y_train, y_test

In [ ]:
class PairDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, pairs_list, labels_list, batch_size=BATCH_SIZE, shuffle=True, **kwargs):
        """
        pairs_list : list of tuples of file paths [(p1, p2), ...]
        labels_list : list of labels [0,1,...]
        """
        super().__init__(**kwargs)  # <-- fix for Keras UserWarning
        self.pairs = pairs_list
        self.labels = np.array(labels_list, dtype=np.float32)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.pairs))
        self.on_epoch_end()

    def __len__(self):
        # Use ceil so partial batches are included
        return int(np.ceil(len(self.pairs) / self.batch_size))

    def __getitem__(self, idx):
        # Ensure we don't exceed the available data
        batch_indices = self.indices[idx*self.batch_size : (idx+1)*self.batch_size]
        if len(batch_indices) == 0:
            raise IndexError(f"No data for batch index {idx}")

        batch_pairs = [self.pairs[i] for i in batch_indices]
        batch_labels = self.labels[batch_indices]

        X1_list, X2_list = [], []
        for p1, p2 in batch_pairs:
            X1_list.append(np.load(p1))
            X2_list.append(np.load(p2))

        X1 = np.stack(X1_list).astype(np.float32)
        X2 = np.stack(X2_list).astype(np.float32)
        y = batch_labels

        return (X1, X2), y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# DL Model Architecture

In [ ]:
# Siamese CNN Model Object Class
class SiameseCNN :
  def __init__(self, input_shape):
    self.input_shape = input_shape
    self.model = self.build()
    self.model.compile(optimizer=tf.keras.optimizers.Adam(),loss='binary_crossentropy',metrics=['accuracy'])

  def build (self) :
    # Shared CNN Encoder
    input_layer = Input(shape=self.input_shape)
    conv1 = Conv2D(64, (3,3), activation='relu')(input_layer)
    pool1 = MaxPooling2D()(conv1)
    conv2 = Conv2D(128, (3,3), activation='relu')(pool1)
    pool2 = MaxPooling2D()(conv2)
    conv3 = Conv2D(256, (3,3), activation='relu')(pool2)
    pool3 = MaxPooling2D()(conv3)
    flat_layer = Flatten()(pool3)
    output_layer = Dense(256, activation='relu')(flat_layer)
    CNNencoder = Model(input_layer,output_layer)

    # Siamese Architecture
    input_A = Input(shape=self.input_shape)
    input_B = Input(shape=self.input_shape)
    encoder_A = CNNencoder(input_A) # Get embeddings for first input image
    encoder_B = CNNencoder(input_B) # Get embeddings for second input image
    distance = Lambda(lambda tensors: tf.abs(tensors[0]-tensors[1]))([encoder_A,encoder_B]) # Compute distance (L1) between embeddings
    output = Dense(1, activation='sigmoid')(distance) # Classwise probability output
    siamese_model = Model(inputs=[input_A,input_B],outputs=output)

    return siamese_model

  # Train the model on Training Data
  def fit (self, train_generator, val_generator, epochs) :
    return self.model.fit(train_generator, validation_data=val_generator, epochs=epochs)

  # Predict output for Input Pair of Images
  def predict (self, X) :
    return self.model.predict(X)

  def summary (self) :
    self.model.summary()

  def save (self, filename) :
    self.model.save(filename)

# Execution Cell

Setup

In [ ]:
# Analyzing and Preprocessing Dataset
preprocess_dataset()

# Create Pair Dataset Lists for pairs and thier labels, and split them into training and testing sets.
X_train, X_test, y_train, y_test = generate_data_pairs_by_path(POSITIVE_PAIRS, NEGATIVE_PAIRS, TEST_SET_RATIO)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=VAL_SET_RATIO, random_state=GLOBAL_RANDOM_STATE)

# # Initialize Data Generator Objects that load dataset image pairs into memory batch-by-batch during training
train_set_generator = PairDataGenerator(X_train,y_train,batch_size=BATCH_SIZE, shuffle=True) # For training, set shuffle=True
val_set_generator = PairDataGenerator(X_val,y_val,batch_size=BATCH_SIZE, shuffle=False) # For validation/test, set shuffle=False
test_set_generator = PairDataGenerator(X_test,y_test,batch_size=BATCH_SIZE, shuffle=False) # For validation/test, set shuffle=False

Training

In [ ]:
# Initialize the Model and Train the Model
model = SiameseCNN((IMAGE_X,IMAGE_Y,1)) # Input Image Format as parameter
model.summary()
training_history = model.fit(train_set_generator, val_set_generator, EPOCHS)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 256)       │ 252,290,3… │ input_layer_1[0]… │
│ (Functional)        │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 256)       │          0 │ functional[0][0], │
│                     │                   │            │ functional[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        257 │ lambda[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 252,290,561 (962.41 MB)

 Trainable params: 252,290,561 (962.41 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
 269/4500 ━━━━━━━━━━━━━━━━━━━━ 45:03 639ms/step - accuracy: 0.8590 - loss: 0.2827

KeyboardInterrupt: 

Testing

In [ ]:
# Testing the Model
y_pred = model.predict(test_set_generator) # Outputs list of probabilities
print(len(X_test))
print(len(y_test))
y_pred = (y_pred >= 0.5).astype(int).flatten() # Convert probabilities to 0/1 (binary classification) based on threshold (0.5)
test_set_generator = PairDataGenerator(X_test, y_test, batch_size=BATCH_SIZE, shuffle=False) # reinitialize generator so we can read labels again
y_test = np.concatenate([y for _, y in test_set_generator]) # Get true labels from the generator
print(f"Accuracy : {accuracy_score(y_test,y_pred):.2f}")

Saving the Trained and Tested Model

In [ ]:
# Save model locally in runtime
model.save("siamese_4kpairs_20epochs.h5")

In [ ]:
# Optional : Save model to drive
from google.colab import drive
drive.mount('/content/drive')
model.save('/content/drive/MyDrive/siamese_4koriginalpairs_20epochs_1stTraining.h5')

# Rough Work

In [ ]:
preprocess_dataset()
total = 0
per_writer = {}
for folder in os.listdir(PREPROCESSED_PATH):
    folder_path = os.path.join(PREPROCESSED_PATH, folder)
    if os.path.isdir(folder_path):
        n = len([f for f in os.listdir(folder_path) if f.endswith(".npy")])
        per_writer[folder] = n
        total += n * (n + 1) // 2

print("Writers counted:", len(per_writer))
print("Total positive pairs (exact):", total)
